# Whisper Fine-Tuning Experiments E4-E6

This Colab notebook completes the remaining model-improvement experiments:

- **E4:** Small Arabic fine-tuned Whisper model
- **E5:** Larger Arabic fine-tuned Whisper model
- **E6:** Fine-tuned Whisper model with augmentation

Use a GPU runtime. This notebook is designed for Google Colab/Kaggle because local CPU fine-tuning is not practical.

## 1. Install Dependencies

In [ ]:
!pip -q install transformers datasets accelerate evaluate jiwer soundfile librosa audiomentations huggingface_hub

## 2. Imports and Configuration

In [ ]:
import re
import random
import numpy as np
import soundfile as sf
import librosa
import torch
import evaluate
from dataclasses import dataclass
from pathlib import Path
import csv
import tarfile
from huggingface_hub import hf_hub_download
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, Gain

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "openai/whisper-small"
LANGUAGE = "Arabic"
TASK = "transcribe"
SAMPLE_RATE = 16000

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Arabic Normalization and Metrics

In [ ]:
AR_DIACRITICS_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
PUNCT_RE = re.compile(r"[^\w\s\u0600-\u06FF]", re.UNICODE)
SPACE_RE = re.compile(r"\s+")


def normalize_arabic(text):
    text = text.strip()
    text = AR_DIACRITICS_RE.sub("", text)
    text = text.replace("ـ", "")
    text = re.sub("[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ة", "ه")
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text)
    return text.strip()

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_text_metrics(predictions, references):
    preds_norm = [normalize_arabic(x) for x in predictions]
    refs_norm = [normalize_arabic(x) for x in references]
    return {
        "wer_raw": wer_metric.compute(predictions=predictions, references=references),
        "wer_normalized": wer_metric.compute(predictions=preds_norm, references=refs_norm),
        "cer_normalized": cer_metric.compute(predictions=preds_norm, references=refs_norm),
    }

## 4. Load Arabic FLEURS Dataset

FLEURS provides Arabic audio with trusted transcripts, which makes WER/CER evaluation valid.

In [ ]:

def load_fleurs_ar_eg_direct(base_dir="fleurs_ar_eg"):
    base_dir = Path(base_dir)
    base_dir.mkdir(parents=True, exist_ok=True)

    splits = {}
    for split_name, hf_split in [("train", "train"), ("validation", "dev"), ("test", "test")]:
        print(f"Downloading FLEURS ar_eg {hf_split} metadata/audio if needed...")

        tsv_path = Path(hf_hub_download(
            repo_id="google/fleurs",
            repo_type="dataset",
            filename=f"data/ar_eg/{hf_split}.tsv",
            local_dir=base_dir,
        ))
        tar_path = Path(hf_hub_download(
            repo_id="google/fleurs",
            repo_type="dataset",
            filename=f"data/ar_eg/audio/{hf_split}.tar.gz",
            local_dir=base_dir,
        ))

        extract_dir = base_dir / "audio" / hf_split
        marker = extract_dir / ".extracted"
        if not marker.exists():
            extract_dir.mkdir(parents=True, exist_ok=True)
            with tarfile.open(tar_path, "r:gz") as tar:
                tar.extractall(extract_dir)
            marker.write_text("ok", encoding="utf-8")

        rows = []
        with tsv_path.open(encoding="utf-8") as f:
            reader = csv.reader(f, delimiter="\t")
            for row in reader:
                if len(row) < 4:
                    continue
                wav_name = row[1]
                raw_text = row[2].strip()
                normalized_text = row[3].strip()
                wav_path = extract_dir / hf_split / wav_name
                if not wav_path.exists():
                    continue
                rows.append({
                    "audio_path": str(wav_path),
                    "transcription": normalized_text or raw_text,
                    "raw_transcription": raw_text,
                    "id": wav_path.stem,
                })

        splits[split_name] = Dataset.from_list(rows)
        print(split_name, splits[split_name])

    return DatasetDict(splits)


dataset = load_fleurs_ar_eg_direct()

print(dataset)
print(dataset["train"][0].keys())
print(dataset["train"][0]["transcription"])


## 5. Choose Dataset Sizes for Structured Experiments

Adjust these sizes depending on GPU time. The defaults are small enough for a course demo.

In [ ]:
SMALL_TRAIN_SIZE = 50
LARGE_TRAIN_SIZE = 150
EVAL_SIZE = 20

train_small = dataset["train"].shuffle(seed=SEED).select(range(SMALL_TRAIN_SIZE))
train_large = dataset["train"].shuffle(seed=SEED).select(range(LARGE_TRAIN_SIZE))
eval_data = dataset["validation"].shuffle(seed=SEED).select(range(EVAL_SIZE))

print(train_small, train_large, eval_data)

## 6. Processor and Model Setup

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

def load_model():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    model.generation_config.language = LANGUAGE.lower()
    model.generation_config.task = TASK
    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=LANGUAGE,
        task=TASK,
    )
    model.generation_config.suppress_tokens = []

    return model

## 7. Preprocessing Functions

In [ ]:
augmenter = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.008, p=0.35),
    TimeStretch(min_rate=0.95, max_rate=1.05, p=0.25),
    Gain(min_gain_db=-3, max_gain_db=3, p=0.35),
])


def load_audio_array(path, target_sr=SAMPLE_RATE):
    """Read audio without datasets.Audio/torchcodec to avoid Kaggle decoder errors."""
    array, sr = sf.read(path, dtype="float32", always_2d=False)
    if array.ndim > 1:
        array = np.mean(array, axis=1)
    if sr != target_sr:
        array = librosa.resample(array, orig_sr=sr, target_sr=target_sr)
    return array.astype(np.float32)


def prepare_example(example, augment=False):
    array = load_audio_array(example["audio_path"], SAMPLE_RATE)
    if augment:
        array = augmenter(samples=array, sample_rate=SAMPLE_RATE)

    input_features = processor.feature_extractor(
        array,
        sampling_rate=SAMPLE_RATE,
    ).input_features[0]

    text = example.get("transcription") or example.get("raw_transcription")
    labels = processor.tokenizer(text).input_ids

    return {
        "input_features": input_features,
        "labels": labels,
        "reference": text,
    }


def preprocess_dataset(ds, augment=False, max_items=None):
    """Manual preprocessing is slower than multiprocessing but much more reliable on Kaggle."""
    rows = []
    total = len(ds) if max_items is None else min(len(ds), max_items)

    for i in range(total):
        example = ds[i]
        try:
            if i % 5 == 0:
                print(f"preprocessing {i + 1}/{total}: {example.get('id', '')}")
            rows.append(prepare_example(example, augment=augment))
        except Exception as exc:
            print(f"[SKIP] {example.get('audio_path', '')}: {exc}")

    print(f"Prepared {len(rows)}/{total} usable examples")
    return Dataset.from_list(rows)


## 8. Data Collator

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 9. Training and Evaluation Helpers

In [ ]:
def compute_metrics(eval_pred):
    pred_ids = eval_pred.predictions
    label_ids = eval_pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return compute_text_metrics(pred_str, label_str)


def train_experiment(name, train_ds, eval_ds, output_dir, epochs=1):
    model = load_model()
    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-5,
        warmup_steps=50,
        num_train_epochs=epochs,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=25,
        predict_with_generate=True,
        eval_accumulation_steps=1,
        generation_max_length=128,
        report_to="none",
        load_best_model_at_end=False,
        
        
        seed=SEED,
    )

    trainer_kwargs = dict(
        args=args,
        model=model,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    import inspect
    trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = processor.feature_extractor
    else:
        trainer_kwargs["tokenizer"] = processor.feature_extractor

    trainer = Seq2SeqTrainer(**trainer_kwargs)

    print(f"Training {name}...")
    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)
    print(name, metrics)
    return metrics

## 10. Prepare Evaluation Data

In [ ]:
eval_prepared = preprocess_dataset(eval_data, augment=False)
print(eval_prepared)

## 11. E4: Small Fine-Tuned Model

In [ ]:
train_small_prepared = preprocess_dataset(train_small, augment=False)

e4_metrics = train_experiment(
    "E4_small_finetune",
    train_small_prepared,
    eval_prepared,
    "outputs/e4_small_finetune",
    epochs=1,
)

print("E4:", e4_metrics)

## 12. E5: Larger Fine-Tuned Model

In [ ]:
train_large_prepared = preprocess_dataset(train_large, augment=False)

e5_metrics = train_experiment(
    "E5_large_finetune",
    train_large_prepared,
    eval_prepared,
    "outputs/e5_large_finetune",
    epochs=1,
)

print("E5:", e5_metrics)

## 13. E6: Fine-Tuned Model With Augmentation

In [ ]:
train_large_augmented = preprocess_dataset(train_large, augment=True)

e6_metrics = train_experiment(
    "E6_augmented_finetune",
    train_large_augmented,
    eval_prepared,
    "outputs/e6_augmented_finetune",
    epochs=1,
)

print("E6:", e6_metrics)

## 14. Completed Results

These are the completed Kaggle GPU results using FLEURS Arabic Egypt with `SMALL_TRAIN_SIZE = 50`, `LARGE_TRAIN_SIZE = 150`, and `EVAL_SIZE = 20`.

| Experiment | Train Samples | Augmentation | Eval Loss | Raw WER | Normalized WER | Normalized CER |
|---|---:|---:|---:|---:|---:|---:|
| E4 Small fine-tune | 50 | No | 2.0174 | 0.2899 | 0.2671 | 0.0675 |
| E5 Larger fine-tune | 150 | No | **1.6712** | **0.2769** | **0.2508** | **0.0618** |
| E6 Larger fine-tune + augmentation | 150 | Yes | 1.6823 | **0.2769** | **0.2508** | **0.0618** |

E5 improved over E4, showing that increasing the Arabic fine-tuning set improved validation performance. E6 matched E5 on WER/CER but had slightly worse validation loss, so this augmentation setup did not provide a measurable improvement in the small run.
